In [103]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, SimpleRNN
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

In [54]:
X = np.loadtxt('../data/processed/X_7steps_encoded.csv', delimiter=',')
y = np.loadtxt('../data/processed/y_7steps_encoded.csv', delimiter=',')

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (211, 131)
y shape: (211,)


In [55]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=5)

print('X Train shape:', X_train.shape)
print('X Test shape:', X_test.shape)
print('y Train shape:', y_train.shape)
print('y Test shape:', y_test.shape)

X Train shape: (189, 131)
X Test shape: (22, 131)
y Train shape: (189,)
y Test shape: (22,)


In [91]:
ann = Sequential([
  Input(shape=(X_train.shape[1],)),
  Dense(32, activation='relu'),
  Dense(128, activation='softmax')
])
ann.summary()

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_42 (Dense)                │ (None, 32)             │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 128)            │         4,224 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,448 (33.00 KB)

 Trainable params: 8,448 (33.00 KB)

 Non-trainable params: 0 (0.00 B)

In [92]:
ann.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(), metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)])
ann.fit(X_train, y_train, epochs=100, validation_split=0.2)

Epoch 1/100


5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - accuracy: 0.0000e+00 - loss: 4.9002 - sparse_top_k_categorical_accuracy: 0.0131 - val_accuracy: 0.0000e+00 - val_loss: 4.7862 - val_sparse_top_k_categorical_accuracy: 0.0000e+00
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0022 - loss: 4.8066 - sparse_top_k_categorical_accuracy: 0.0699 - val_accuracy: 0.0000e+00 - val_loss: 4.6858 - val_sparse_top_k_categorical_accuracy: 0.1053
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0332 - loss: 4.7022 - sparse_top_k_categorical_accuracy: 0.1099 - val_accuracy: 0.0000e+00 - val_loss: 4.5879 - val_sparse_top_k_categorical_accuracy: 0.2368
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0877 - loss: 4.6001 - sparse_top_k_categorical_accuracy: 0.1471 - val_accuracy: 0.0789 - val_loss: 4.4902 - val_sparse_top_k_categorical_accuracy: 0.2632
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0921 - loss: 4.5031 - sparse_top_k_categorical_accuracy:

In [93]:
ann.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - accuracy: 0.2727 - loss: 2.4241 - sparse_top_k_categorical_accuracy: 0.5909


[2.424117088317871, 0.27272728085517883, 0.5909090638160706]

### Tuning Hyperparameters

In [104]:
def build_model(hp):
  model = Sequential()
  model.add(Input(shape=(131,)))

  # Hidden Layer 1
  model.add(Dense(
    units=hp.Int('units1', min_value=16, max_value=128, step=16),
    activation='relu',
  ))

  # Dropout Layer 1 (Optional)
  if hp.Boolean('use_dropout1_layer'):
    model.add(Dropout(hp.Float('dropout1', min_value=0.1, max_value=0.5, step=0.1)))

  # Hidden Layer 2 (Optional)
  if hp.Boolean('use_hidden_layer2'):
    model.add(Dense(
      units=hp.Int('units2', min_value=16, max_value=128, step=16),
      activation='relu'
    ))

  # Dropout Layer 2 (Optional)
  if hp.Boolean('use_dropout2_layer'):
    model.add(Dropout(hp.Float('dropout2', min_value=0.1, max_value=0.5, step=0.1)))

  # Output Layer
  model.add(Dense(128, activation='softmax'))

  # Optimizer and Learning Rate
  lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
  optimizer = Adam(learning_rate=lr)

  model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)]
  )

  return model

In [105]:
tuner = kt.RandomSearch(
  build_model,
  objective='val_accuracy',
  max_trials=10,
  executions_per_trial=1,
  directory='../tuning_dir',
  project_name='hero_draft_ann'
)

tuner.search(X_train, y_train, epochs=100, validation_split=0.2)

Trial 10 Complete [00h 00m 17s]
val_accuracy: 0.1315789520740509

Best val_accuracy So Far: 0.5
Total elapsed time: 00h 03m 15s


In [107]:
best_model = tuner.get_best_models()[0]
best_hps = tuner.get_best_hyperparameters()[0]

print("Best Hyperparameters:")
print(f"Units1: {best_hps['units1']}")
print(f"Use Dropout1 Layer: {best_hps['use_dropout1_layer']}")
if best_hps['use_dropout1_layer']:
    print(f"Dropout1: {best_hps['dropout1']}")
print(f"Use Second Layer: {best_hps['use_hidden_layer2']}")
if best_hps['use_hidden_layer2']:
    print(f"Units2: {best_hps['units2']}")
print(f"Use Dropout2 Layer: {best_hps['use_dropout2_layer']}")
if best_hps['use_dropout2_layer']:
    print(f"Dropout2: {best_hps['dropout2']}")
print(f"Learning Rate: {best_hps['learning_rate']}")

Best Hyperparameters:
Units1: 128
Use Dropout1 Layer: False
Use Second Layer: True
Units2: 128
Use Dropout2 Layer: False
Learning Rate: 0.001


In [108]:
best_model.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 749ms/step - accuracy: 0.5000 - loss: 2.3062 - sparse_top_k_categorical_accuracy: 0.8182


[2.306213617324829, 0.5, 0.8181818127632141]

In [109]:
best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]

print("Best trial ID:", best_trial.trial_id)
print("Best val_accuracy:", best_trial.score)
print("Best hyperparameters:", best_trial.hyperparameters.values)


Best trial ID: 04
Best val_accuracy: 0.5
Best hyperparameters: {'units1': 128, 'use_dropout1_layer': False, 'use_hidden_layer2': True, 'use_dropout2_layer': False, 'learning_rate': 0.001, 'dropout1': 0.30000000000000004, 'units2': 128, 'dropout2': 0.30000000000000004}
